# 🩺 Machine Learning — Diagnóstico de Câncer de Mama
**FIAP Postech — IA para Devs | Módulo Machine Learning**

---

## Objetivo
Construir um modelo de classificação para prever se um tumor é **maligno (0)** ou **benigno (1)** com base em características clínicas, utilizando o dataset Wisconsin Breast Cancer.

## Pipeline deste notebook
1. Carregamento e exploração dos dados (EDA)
2. Pré-processamento e Feature Scaling
3. Treinamento e comparação de modelos
4. Avaliação com métricas completas (Recall como principal)
5. Curva ROC e AUC Score
6. Explicabilidade com SHAP

---
> **Por que Recall é a métrica principal?**  
> Em diagnóstico oncológico, um **falso negativo** (classificar maligno como benigno) tem custo muito maior do que um falso positivo. Portanto, minimizamos os falsos negativos maximizando o **Recall**.

## 0. Instalação de dependências

In [ ]:
# Execute apenas se necessário
# !pip install shap matplotlib seaborn scikit-learn pandas numpy

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Dataset
from sklearn.datasets import load_breast_cancer

# Pré-processamento
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Métricas
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    ConfusionMatrixDisplay
)

# Explicabilidade
import shap

# Estilo visual
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

print('✅ Imports concluídos')

---
## 2. Carregamento e Exploração dos Dados (EDA)

In [ ]:
# Carregando o dataset
data = load_breast_cancer()

df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target  # 0 = maligno, 1 = benigno

print('📊 Shape do dataset:', df.shape)
print('\n🏷️ Classes:', dict(zip([0, 1], data.target_names)))
print('\n📋 Features:', list(data.feature_names))

In [ ]:
# Primeiras linhas
df.head()

In [ ]:
# Informações gerais
df.info()

In [ ]:
# Estatísticas descritivas
df.describe().T

In [ ]:
# Verificando valores nulos
nulls = df.isnull().sum()
print('🔍 Valores nulos por coluna:')
print(nulls[nulls > 0] if nulls.sum() > 0 else '✅ Nenhum valor nulo encontrado!')

In [ ]:
# Distribuição das classes
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Contagem
target_counts = df['target'].value_counts()
target_labels = ['Maligno (0)', 'Benigno (1)']
axes[0].bar(target_labels, target_counts.values, color=['#e74c3c', '#2ecc71'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribuição das Classes', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Quantidade')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

# Proporção
axes[1].pie(target_counts.values, labels=target_labels,
            colors=['#e74c3c', '#2ecc71'], autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Proporção das Classes', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.suptitle('Balanceamento do Dataset', y=1.02, fontsize=14, fontweight='bold')
plt.show()

print(f'\nRatio Maligno/Benigno: {target_counts[0]}/{target_counts[1]} ({target_counts[0]/len(df)*100:.1f}% / {target_counts[1]/len(df)*100:.1f}%)')

In [ ]:
# Distribuição das features por classe — primeiras 9 features (mean)
mean_features = [col for col in df.columns if 'mean' in col]

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(mean_features[:9]):
    ax = axes[i]
    df[df['target'] == 0][col].hist(ax=ax, alpha=0.6, color='#e74c3c', label='Maligno', bins=25)
    df[df['target'] == 1][col].hist(ax=ax, alpha=0.6, color='#2ecc71', label='Benigno', bins=25)
    ax.set_title(col.replace(' ', '\n'), fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle('Distribuição das Features (Mean) por Classe', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Mapa de correlação das features 'mean'
corr_features = df[mean_features + ['target']]

plt.figure(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_features.corr(), dtype=bool))
sns.heatmap(
    corr_features.corr(),
    annot=True, fmt='.2f', cmap='RdYlGn',
    mask=mask, linewidths=0.5,
    vmin=-1, vmax=1, center=0,
    annot_kws={'size': 8}
)
plt.title('Mapa de Correlação — Features Mean + Target', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Correlação com o target — ranking
corr_target = df.corr()['target'].drop('target').abs().sort_values(ascending=False)

plt.figure(figsize=(10, 8))
colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in df.corr()['target'].drop('target').loc[corr_target.index]]
corr_target.plot(kind='barh', color=colors)
plt.axvline(0.5, color='gray', linestyle='--', alpha=0.7, label='Limiar 0.5')
plt.title('Correlação Absoluta das Features com o Target', fontsize=13, fontweight='bold')
plt.xlabel('|Correlação|')
plt.legend()
plt.tight_layout()
plt.show()

print('\n🔝 Top 5 features mais correlacionadas com o target:')
print(corr_target.head())

---
## 3. Pré-processamento

In [ ]:
# Separando features e target
X = df.drop('target', axis=1)
y = df['target']

# Split estratificado — garante proporção das classes em train e test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # ← importante para datasets desbalanceados
)

print(f'Train: {X_train.shape[0]} amostras | Test: {X_test.shape[0]} amostras')
print(f'\nDistribuição no Train:\n{y_train.value_counts()}')
print(f'\nDistribuição no Test:\n{y_test.value_counts()}')

In [ ]:
# Scaler — padroniza as features (importante para KNN, SVM, Regressão Logística)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # ← fit apenas no train!

# Visualizando o efeito do scaling
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

pd.DataFrame(X_train_scaled, columns=X.columns)[mean_features[:5]].boxplot(ax=axes[0])
axes[0].set_title('Após StandardScaler', fontsize=11, fontweight='bold')
axes[0].set_xticklabels([c.replace('mean ', '') for c in mean_features[:5]], rotation=15)

X_train[mean_features[:5]].boxplot(ax=axes[1])
axes[1].set_title('Antes do Scaling', fontsize=11, fontweight='bold')
axes[1].set_xticklabels([c.replace('mean ', '') for c in mean_features[:5]], rotation=15)

plt.suptitle('Efeito do StandardScaler nas Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Treinamento e Comparação de Modelos

> Usamos `Pipeline` do sklearn para garantir que o scaler seja aplicado corretamente dentro do cross-validation (evita data leakage).

In [ ]:
# Definindo os pipelines
pipelines = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(random_state=42, max_iter=1000))
    ]),
    'Decision Tree': Pipeline([
        ('model', DecisionTreeClassifier(random_state=42, max_depth=5))
    ]),
    'Random Forest': Pipeline([
        ('model', RandomForestClassifier(n_estimators=100, random_state=42))
    ]),
    'Gradient Boosting': Pipeline([
        ('model', GradientBoostingClassifier(n_estimators=100, random_state=42))
    ]),
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(probability=True, random_state=42))
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ])
}

# Cross-validation estratificada — 5 folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

resultados = {}

print('🔄 Rodando Cross-Validation (5 folds) para cada modelo...\n')

for nome, pipeline in pipelines.items():
    scores_recall   = cross_val_score(pipeline, X, y, cv=cv, scoring='recall')
    scores_f1       = cross_val_score(pipeline, X, y, cv=cv, scoring='f1')
    scores_accuracy = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')
    scores_auc      = cross_val_score(pipeline, X, y, cv=cv, scoring='roc_auc')

    resultados[nome] = {
        'Recall (mean)':   scores_recall.mean(),
        'Recall (std)':    scores_recall.std(),
        'F1 (mean)':       scores_f1.mean(),
        'Accuracy (mean)': scores_accuracy.mean(),
        'AUC (mean)':      scores_auc.mean()
    }

    print(f'✅ {nome}: Recall={scores_recall.mean():.3f} ± {scores_recall.std():.3f} | F1={scores_f1.mean():.3f} | AUC={scores_auc.mean():.3f}')

df_resultados = pd.DataFrame(resultados).T
print('\n📊 Tabela de Resultados:')
df_resultados.sort_values('Recall (mean)', ascending=False)

In [ ]:
# Visualização comparativa
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metricas_plot = ['Recall (mean)', 'F1 (mean)', 'AUC (mean)']
cores = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12', '#1abc9c']

for i, metrica in enumerate(metricas_plot):
    vals = df_resultados[metrica].sort_values(ascending=False)
    axes[i].barh(vals.index, vals.values, color=cores[:len(vals)], edgecolor='white')
    axes[i].set_xlim(0.8, 1.0)
    axes[i].set_title(metrica, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Score')
    for j, (idx, val) in enumerate(vals.items()):
        axes[i].text(val + 0.001, j, f'{val:.3f}', va='center', fontsize=9)

plt.suptitle('Comparação de Modelos — Cross-Validation 5-Fold', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Avaliação do Melhor Modelo no Test Set

In [ ]:
# Selecionando o melhor modelo por Recall
melhor_modelo_nome = df_resultados['Recall (mean)'].idxmax()
print(f'🏆 Melhor modelo por Recall: {melhor_modelo_nome}')

# Treinando no conjunto completo de treino
melhor_pipeline = pipelines[melhor_modelo_nome]
melhor_pipeline.fit(X_train, y_train)

# Predições
y_pred = melhor_pipeline.predict(X_test)
y_proba = melhor_pipeline.predict_proba(X_test)[:, 1]

In [ ]:
# Classification Report
print('=' * 55)
print(f'  Classification Report — {melhor_modelo_nome}')
print('=' * 55)
print(classification_report(y_test, y_pred, target_names=['Maligno', 'Benigno']))

print('\n💡 Interpretação das métricas:')
print('  Precision: dos que o modelo marcou como positivo, quantos realmente são?')
print('  Recall:    dos que realmente são positivos, quantos o modelo encontrou?')
print('  F1-Score:  média harmônica entre Precision e Recall')
print('  Support:   número de amostras reais de cada classe no test set')

In [ ]:
# Matriz de Confusão
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Valores absolutos
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Maligno', 'Benigno'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'Matriz de Confusão\n{melhor_modelo_nome}', fontweight='bold')

# Normalizada
cm_norm = confusion_matrix(y_test, y_pred, normalize='true')
disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=['Maligno', 'Benigno'])
disp_norm.plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title(f'Matriz de Confusão Normalizada\n{melhor_modelo_nome}', fontweight='bold')

plt.tight_layout()
plt.show()

# Destacando falsos negativos
fn = cm[1][0]  # real=benigno, pred=maligno (confusão de label)
fp = cm[0][1]
print(f'\n⚠️  Falsos Negativos (maligno classificado como benigno): {cm[0][1]}')
print(f'   Falsos Positivos (benigno classificado como maligno): {cm[1][0]}')
print(f'\n   Em contexto clínico, FN é o erro mais crítico.')

In [ ]:
# Curva ROC e AUC
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc_score = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#3498db', lw=2, label=f'ROC Curve (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1.5, label='Random Classifier (AUC = 0.500)')
plt.fill_between(fpr, tpr, alpha=0.1, color='#3498db')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Especificidade)', fontsize=11)
plt.ylabel('True Positive Rate (Recall / Sensibilidade)', fontsize=11)
plt.title(f'Curva ROC — {melhor_modelo_nome}', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\n🎯 AUC Score: {auc_score:.4f}')
print('   AUC = 1.0 → classificação perfeita')
print('   AUC = 0.5 → equivale a chute aleatório')

In [ ]:
# Curvas ROC de todos os modelos sobrepostas
plt.figure(figsize=(9, 7))
plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random (AUC=0.50)')

for nome, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    proba = pipeline.predict_proba(X_test)[:, 1]
    fpr_m, tpr_m, _ = roc_curve(y_test, proba)
    auc_m = roc_auc_score(y_test, proba)
    plt.plot(fpr_m, tpr_m, lw=2, label=f'{nome} (AUC={auc_m:.3f})')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('Comparação de Curvas ROC — Todos os Modelos', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Explicabilidade com SHAP

SHAP (SHapley Additive exPlanations) quantifica a contribuição de cada feature para cada predição individual, baseado na teoria dos jogos cooperativos (Shapley values).

In [ ]:
# Usando Random Forest para SHAP (Tree-based)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Criando o explainer
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test)

print('✅ SHAP values calculados')
print(f'   Shape: {np.array(shap_values).shape}  →  [classes, amostras, features]')

In [ ]:
# Summary Plot — importância global das features
shap.summary_plot(
    shap_values[1],  # classe 1 = benigno
    X_test,
    feature_names=list(X.columns),
    plot_type='bar',
    max_display=15,
    show=False
)
plt.title('SHAP — Importância Global das Features (Top 15)', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Beeswarm plot — distribuição dos SHAP values por feature
shap.summary_plot(
    shap_values[1],
    X_test,
    feature_names=list(X.columns),
    max_display=15,
    show=False
)
plt.title('SHAP Beeswarm — Impacto e Direção das Features', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('\n💡 Leitura do gráfico:')
print('   Cor vermelha  → valor alto da feature')
print('   Cor azul      → valor baixo da feature')
print('   Posição X     → impacto na predição (+ = empurra para benigno)')

In [ ]:
# Force plot — explicando uma predição individual
idx = 0  # Primeira amostra do test set

print(f'📍 Explicando amostra #{idx}')
print(f'   Real:     {"Benigno" if y_test.iloc[idx] == 1 else "Maligno"}')
print(f'   Predito:  {"Benigno" if rf_model.predict(X_test.iloc[[idx]])[0] == 1 else "Maligno"}')
print(f'   Probabilidade (benigno): {rf_model.predict_proba(X_test.iloc[[idx]])[0][1]:.3f}')

shap.force_plot(
    explainer.expected_value[1],
    shap_values[1][idx],
    X_test.iloc[idx],
    feature_names=list(X.columns),
    matplotlib=True,
    show=False
)
plt.title(f'SHAP Force Plot — Amostra #{idx}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance comparando: RF nativo vs SHAP
rf_importance = pd.Series(rf_model.feature_importances_, index=X.columns)
shap_importance = pd.Series(np.abs(shap_values[1]).mean(axis=0), index=X.columns)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

rf_importance.nlargest(15).sort_values().plot(kind='barh', ax=axes[0], color='#3498db')
axes[0].set_title('Feature Importance\n(Random Forest nativo)', fontweight='bold')
axes[0].set_xlabel('Importância')

shap_importance.nlargest(15).sort_values().plot(kind='barh', ax=axes[1], color='#e67e22')
axes[1].set_title('Feature Importance\n(SHAP mean |Shapley value|)', fontweight='bold')
axes[1].set_xlabel('Mean |SHAP value|')

plt.suptitle('Comparação: Importância RF vs SHAP', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Resumo Executivo

In [ ]:
from sklearn.metrics import recall_score, f1_score, accuracy_score

print('=' * 60)
print('           RESUMO DO ESTUDO — FIAP ML Challenge')
print('=' * 60)
print(f'\n📦 Dataset: Wisconsin Breast Cancer')
print(f'   {len(df)} amostras | 30 features | 2 classes')
print(f'   Maligno: {(y==0).sum()} | Benigno: {(y==1).sum()}')
print(f'\n🎯 Métrica principal: Recall')
print(f'   Justificativa: minimizar falsos negativos em diagnóstico oncológico')
print(f'\n📊 Comparação dos modelos (Cross-Validation 5-Fold):')
print(df_resultados[['Recall (mean)', 'F1 (mean)', 'AUC (mean)']]
      .sort_values('Recall (mean)', ascending=False)
      .to_string())

# Avaliando no test set
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print(f'\n🏆 Modelo final avaliado no test set: Random Forest')
print(f'   Recall:   {recall_score(y_test, y_pred_rf):.4f}')
print(f'   F1-Score: {f1_score(y_test, y_pred_rf):.4f}')
print(f'   Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'   AUC:      {roc_auc_score(y_test, y_proba_rf):.4f}')
print(f'\n🔍 Explicabilidade: SHAP (TreeExplainer)')
top3_shap = shap_importance.nlargest(3).index.tolist()
print(f'   Top 3 features mais influentes:')
for i, feat in enumerate(top3_shap, 1):
    print(f'   {i}. {feat}')
print('\n' + '=' * 60)

---
## 8. Próximos Passos

Para aprofundar e ir além do Tech Challenge:

| Técnica | Por que explorar? |
|---|---|
| **GridSearchCV / RandomizedSearchCV** | Otimização de hiperparâmetros dos modelos |
| **SMOTE** | Técnica de oversampling para datasets desbalanceados |
| **Threshold tuning** | Ajustar o limiar de decisão para maximizar Recall |
| **Redução de dimensionalidade (PCA)** | Visualizar clusters e reduzir 30→2 features |
| **LIME** | Alternativa ao SHAP para explicabilidade local |
| **MLflow** | Registro e rastreamento de experimentos |

---
*Notebook criado para FIAP Postech — IA para Devs | Módulo Machine Learning*